In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchinfo
import torch.amp as amp

from Generator import noise_mul, noise_add
from Helper import load_checkpoint_generic, save_checkpoint_generic
from Model import EdgePreservingDenoiser, SyntheticNoiseDataset, evaluate_model
from Model import load_dataset_arrays, save_dataset_arrays

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [2]:
# =============================================================================
# DEVICE SETUP
# =============================================================================
print("=" * 80)
print("DEVICE CONFIGURATION")
print("=" * 80)

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    print(f"✅ CUDA available with {num_devices} device(s)")

    for i in range(num_devices):
        device_name = torch.cuda.get_device_name(i)
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9  # in GB
        print(f"   Device {i}: {device_name} ({total_memory:.2f} GB)")
else:
    print("⚠️  CUDA not available, using CPU")

print("=" * 80)
print()

DEVICE CONFIGURATION
✅ CUDA available with 1 device(s)
   Device 0: NVIDIA GeForce RTX 3070 (8.59 GB)



In [3]:
# =============================================================================
# HYPERPARAMETERS
# =============================================================================
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_ITERATIONS = 100_000  # Total training iterations
VALIDATION_INTERVAL = 100  # Validate every N iterations
CHECKPOINT_INTERVAL = 1000  # Save checkpoint every N iterations
LOG_INTERVAL = 20  # Log training progress every N iterations

# Dataset Configuration
TRAIN_SIZE = 5000  # Number of training samples
VAL_SIZE = 500     # Number of validation samples

# Noise Configuration
# NOISE_FN = noise_add  # Additive noise
NOISE_FN = noise_mul   # Multiplicative noise
NOISE_RANGE = (0.1, 0.3)

# Loss Configuration
LOSS_FN = 'MSE'  # Options: MSE, RMSE, MAE

# Checkpoint directory
CHECKPOINT_DIR = f"checkpoints-{NOISE_FN.__name__}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


In [4]:
# =============================================================================
# MODEL SETUP & DATA LOADING
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 80)
print(f"TRAINING DEVICE: {device}")
print("=" * 80)
print()

# Model
model = EdgePreservingDenoiser().to(device)
if device.type == 'cuda':
    model.to(memory_format=torch.channels_last)

# === LOAD OR GENERATE TRAIN DATASET ===
print("=" * 80)
print("DATASET PREPARATION")
print("=" * 80)
train_clean, train_noisy = load_dataset_arrays(CHECKPOINT_DIR, "train")

if train_clean is not None:
    # Podatki obstajajo - uporabi jih (že v CHW formatu)
    train_dataset = SyntheticNoiseDataset(
        clean_images=train_clean,
        noisy_images=train_noisy,
        to_channels_last=True,
    )
    print(f"✅ Train dataset created from loaded data: {len(train_dataset)} samples")
else:
    # Podatki ne obstajajo - generiraj nove
    print(f"🎨 Generating train dataset ({TRAIN_SIZE} samples)...")
    train_dataset = SyntheticNoiseDataset(
        num_samples=TRAIN_SIZE,
        noise_fn=NOISE_FN,
        noise_range=NOISE_RANGE,
        to_channels_last=True,
    )
    # Shrani dataset direktno (že v CHW formatu)
    save_dataset_arrays(CHECKPOINT_DIR, "train", train_dataset)
    print(f"✅ Train dataset generated and saved: {len(train_dataset)} samples")

# === LOAD OR GENERATE VALIDATION DATASET ===
val_clean, val_noisy = load_dataset_arrays(CHECKPOINT_DIR, "val")

if val_clean is not None:
    # Podatki obstajajo - uporabi jih (že v CHW formatu)
    val_dataset = SyntheticNoiseDataset(
        clean_images=val_clean,
        noisy_images=val_noisy,
        to_channels_last=True,
    )
    print(f"✅ Validation dataset created from loaded data: {len(val_dataset)} samples")
else:
    # Podatki ne obstajajo - generiraj nove
    print(f"🎨 Generating validation dataset ({VAL_SIZE} samples)...")
    val_dataset = SyntheticNoiseDataset(
        num_samples=VAL_SIZE,
        noise_fn=NOISE_FN,
        noise_range=NOISE_RANGE,
        to_channels_last=True,
    )
    # Shrani dataset direktno (že v CHW formatu)
    save_dataset_arrays(CHECKPOINT_DIR, "val", val_dataset)
    print(f"✅ Validation dataset generated and saved: {len(val_dataset)} samples")

torch.backends.cudnn.benchmark = True

# === CREATE DATALOADERS ===
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)

print("=" * 80)
print()


TRAINING DEVICE: cuda

DATASET PREPARATION
📂 Loading train dataset from checkpoints-noise_mul\train_dataset.pkl
✅ Loaded 5000 samples
✅ Train dataset created from loaded data: 5000 samples
📂 Loading val dataset from checkpoints-noise_mul\val_dataset.pkl
✅ Loaded 500 samples
✅ Validation dataset created from loaded data: 500 samples



In [5]:
# =============================================================================
# OPTIMIZER & LOSS FUNCTION
# =============================================================================
# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = amp.GradScaler()


# Loss function
if LOSS_FN == 'MSE':
    criterion = nn.MSELoss()
elif LOSS_FN == 'RMSE':
    criterion = lambda pred, target: torch.sqrt(nn.MSELoss()(pred, target))
elif LOSS_FN == 'MAE':
    criterion = nn.L1Loss()
else:
    raise ValueError(f"Unknown loss function: {LOSS_FN}")

print("=" * 80)
print("OPTIMIZER & LOSS")
print("=" * 80)
print(f"Optimizer:      AdamW")
print(f"Learning Rate:  {LEARNING_RATE}")
print(f"Loss Function:  {LOSS_FN}")
print("=" * 80)
print()

OPTIMIZER & LOSS
Optimizer:      AdamW
Learning Rate:  0.001
Loss Function:  MSE



In [6]:
# =============================================================================
# MODEL ARCHITECTURE SUMMARY
# =============================================================================
torchinfo.summary(model, input_size=(1, 3, 256, 256), device=device)

Layer (type:depth-idx)                   Output Shape              Param #
EdgePreservingDenoiser                   [1, 3, 256, 256]          --
├─FilterBranch: 1-1                      [1, 3, 8, 256, 256]       --
│    └─Conv2d: 2-1                       [1, 8, 256, 256]          976
│    └─Conv2d: 2-2                       [1, 8, 256, 256]          976
│    └─Conv2d: 2-3                       [1, 8, 256, 256]          976
├─WeightBranch: 1-2                      [1, 8, 256, 256]          --
│    └─ResNetBlock: 2-4                  [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                  [1, 32, 256, 256]         128
│    │    └─Conv2d: 3-2                  [1, 32, 256, 256]         896
│    │    └─Dropout2d: 3-3               [1, 32, 256, 256]         --
│    │    └─ReLU: 3-4                    [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-5                  [1, 32, 256, 256]         9,248
│    │    └─Dropout2d: 3-6               [1, 32, 256, 256]         --
│    │ 

In [7]:
# =============================================================================
# TRAINING LOOP
# =============================================================================
# Load checkpoint
checkpoint = load_checkpoint_generic(CHECKPOINT_DIR, device=device)

start_iteration = 1
train_losses = []
val_losses = []
val_snrs = []
val_psnrs = []
best_val_loss = float('inf')  # Track best validation loss

if checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_iteration = checkpoint.get('iteration', 0) + 1
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    val_snrs = checkpoint.get('val_snrs', [])
    val_psnrs = checkpoint.get('val_psnrs', [])
    best_val_loss = checkpoint.get('best_val_loss', min(val_losses) if val_losses else float('inf'))

    print("=" * 80)
    print("CHECKPOINT LOADED")
    print("=" * 80)
    print(f"Resuming from iteration:    {start_iteration}")
    print(f"Train losses recorded:      {len(train_losses)}")
    print(f"Val losses recorded:        {len(val_losses)}")
    if val_losses:
        print(f"Best Val Loss so far:       {best_val_loss:.6f}")
    if val_psnrs:
        print(f"Best Val PSNR so far:       {max(val_psnrs):.2f} dB")
    print("=" * 80)
    print()
else:
    print("=" * 80)
    print("STARTING NEW TRAINING")
    print("=" * 80)
    print()

model.train()

# Iterator for infinite iteration through dataset
train_iterator = iter(train_loader)

print("=" * 80)
print("TRAINING STARTED")
print("=" * 80)
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print()

training_start_time = time.time()
last_log_time = time.time()

for iteration in range(start_iteration, MAX_ITERATIONS + 1):
    try:
        noisy, clean = next(train_iterator)
    except StopIteration:
        # Restart iterator when we reach the end of dataset
        train_iterator = iter(train_loader)
        noisy, clean = next(train_iterator)

    noisy = noisy.to(device, non_blocking=True)
    clean = clean.to(device, non_blocking=True)

    # Forward pass
    optimizer.zero_grad(set_to_none=True)
    with amp.autocast(device_type=device.type):
        denoised = model(noisy)
        loss = criterion(denoised, clean)

    # Backward pass
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    # Track metrics
    train_losses.append(loss.item())

    # === CONSOLE LOGGING ===
    if iteration % LOG_INTERVAL == 0 or iteration == 1 or iteration == MAX_ITERATIONS:
        current_time = time.time()
        interval_time = current_time - last_log_time

        # Calculate actual iterations in this interval
        if iteration == 1:
            actual_iters = 1
        elif iteration == MAX_ITERATIONS and iteration % LOG_INTERVAL != 0:
            actual_iters = iteration % LOG_INTERVAL
        else:
            actual_iters = LOG_INTERVAL

        avg_iter_time = interval_time / actual_iters
        last_log_time = current_time

        # Recent average loss
        recent_losses = train_losses[-LOG_INTERVAL:] if len(train_losses) >= LOG_INTERVAL else train_losses
        avg_loss = np.mean(recent_losses)

        # Progress percentage
        progress = (iteration / MAX_ITERATIONS) * 100

        # Calculate ETA using OVERALL average (not just recent interval)
        # This gives more accurate estimates, especially later in training
        elapsed_total = current_time - training_start_time
        completed_iters = iteration - start_iteration + 1
        overall_avg_iter_time = elapsed_total / completed_iters if completed_iters > 0 else avg_iter_time

        remaining_iters = MAX_ITERATIONS - iteration
        eta_seconds = remaining_iters * overall_avg_iter_time
        eta_hours = eta_seconds / 3600

        print(f"[Iter {iteration:6d}/{MAX_ITERATIONS}] "
              f"Loss: {loss.item():.6f} (avg: {avg_loss:.6f}) | "
              f"Interval: {interval_time:.2f}s ({avg_iter_time:.3f}s/iter) | "
              f"ETA: {eta_hours:.2f}h | "
              f"Progress: {progress:.1f}%")

    # === VALIDATION ===
    if iteration % VALIDATION_INTERVAL == 0:
        val_start = time.time()
        val_loss, val_snr, val_psnr = evaluate_model(model, val_loader, device)
        val_time = time.time() - val_start

        val_losses.append(val_loss)
        val_snrs.append(val_snr)
        val_psnrs.append(val_psnr)

        # Check if this is the best model based on validation loss
        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        # Compact one-line validation output
        best_indicator = " 🌟 BEST!" if is_best else ""
        print(f"[VAL @ {iteration:6d}] Loss: {val_loss:.6f}{best_indicator} | "
              f"SNR: {val_snr:.2f} dB | PSNR: {val_psnr:.2f} dB | "
              f"Time: {val_time:.2f}s")

        model.train()  # Back to training mode

    # === CHECKPOINT SAVING ===
    if iteration % CHECKPOINT_INTERVAL == 0 or iteration == MAX_ITERATIONS:
        checkpoint_data = {
            'iteration': iteration,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_snrs': val_snrs,
            'val_psnrs': val_psnrs,
            'best_val_loss': best_val_loss,
            'config': {
                'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE,
                'noise_fn': NOISE_FN.__name__,
                'loss_fn': LOSS_FN,
                'train_size': TRAIN_SIZE,
                'val_size': VAL_SIZE,
                'noise_range': NOISE_RANGE
            }
        }

        save_checkpoint_generic(
            CHECKPOINT_DIR,
            iteration,
            checkpoint_data,
            max_checkpoints=5
        )

        # Save best model separately when we have a new best
        if val_losses and val_losses[-1] == best_val_loss:
            best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
            torch.save(model.state_dict(), best_model_path)
            print(f"💾 Best model saved: {best_model_path}")

# Save final model
final_model_path = f"denoiser_{NOISE_FN.__name__}_iter{MAX_ITERATIONS}.pth"
torch.save(model.state_dict(), final_model_path)

# Training summary
total_time = time.time() - training_start_time
total_hours = total_time / 3600

print("\n" + "=" * 80)
print("TRAINING COMPLETE!")
print("=" * 80)
print(f"Total Training Time:    {total_hours:.2f} hours ({total_time:.0f}s)")
print(f"Total Iterations:       {MAX_ITERATIONS - start_iteration + 1}")
print(f"Final Train Loss:       {train_losses[-1]:.6f}")
print(f"Final Val Loss:         {val_losses[-1]:.6f}")
print(f"Best Val SNR:           {max(val_snrs):.2f} dB")
print(f"Best Val PSNR:          {max(val_psnrs):.2f} dB")
print(f"Final Model Saved:      {final_model_path}")
print("=" * 80)

✅ Loaded checkpoint: checkpoints-noise_mul\checkpoint_iteration_41000.pth (iteration 41000)
CHECKPOINT LOADED
Resuming from iteration:    41001
Train losses recorded:      41000
Val losses recorded:        410
Best Val Loss so far:       0.000400
Best Val PSNR so far:       33.82 dB

TRAINING STARTED
Start Time: 2025-12-14 23:00:00

[Iter  41020/100000] Loss: 0.000434 (avg: 0.000460) | Interval: 8.05s (0.403s/iter) | ETA: 6.60h | Progress: 41.0%
[Iter  41040/100000] Loss: 0.000433 (avg: 0.000468) | Interval: 3.70s (0.185s/iter) | ETA: 4.81h | Progress: 41.0%
[Iter  41060/100000] Loss: 0.000440 (avg: 0.000458) | Interval: 3.68s (0.184s/iter) | ETA: 4.21h | Progress: 41.1%
[Iter  41080/100000] Loss: 0.000434 (avg: 0.000470) | Interval: 3.70s (0.185s/iter) | ETA: 3.91h | Progress: 41.1%
[Iter  41100/100000] Loss: 0.000473 (avg: 0.000457) | Interval: 3.68s (0.184s/iter) | ETA: 3.73h | Progress: 41.1%
[VAL @  41100] Loss: 0.000404 | SNR: 28.94 dB | PSNR: 33.74 dB | Time: 5.45s
[Iter  41120/